<a href="https://colab.research.google.com/github/AksharaBasu/GenAI/blob/main/LLM5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 3-LLM GENERATE → CHECK → CORRECT → BACKUP SYSTEM
# ============================================================

# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q -U langgraph langchain-openrouter langchain-groq gradio


# ============================================================
# 2. IMPORTS
# ============================================================

import gradio as gr
from typing import TypedDict

from google.colab import userdata

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# ============================================================
# 3. GET API KEYS
# ============================================================

OPENROUTER_API_KEY = userdata.get("openrouter")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
AKSHARA_API_KEY = userdata.get("akshara")


print("=" * 70)
print("API KEY STATUS")
print("=" * 70)

print(
    "openrouter   :",
    "✅ Found" if OPENROUTER_API_KEY else "❌ Missing"
)

print(
    "GROQ_API_KEY :",
    "✅ Found" if GROQ_API_KEY else "❌ Missing"
)

print(
    "akshara      :",
    "✅ Found" if AKSHARA_API_KEY else "❌ Missing"
)

print("=" * 70)


# ============================================================
# 4. CREATE LLMs
# ============================================================

# ------------------------------------------------------------
# LLM 1
# Generates the first answer
# ------------------------------------------------------------

llm1 = ChatOpenRouter(
    api_key=OPENROUTER_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)


# ------------------------------------------------------------
# LLM 2
# Checks LLM 1 answer and corrects it if needed
# ------------------------------------------------------------

llm2 = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)


# ------------------------------------------------------------
# LLM 3
# Emergency backup
# ------------------------------------------------------------

llm3 = ChatOpenRouter(
    api_key=AKSHARA_API_KEY,
    model="qwen/qwen3-32b",
    temperature=0.2,
    max_tokens=2048
)


print("✅ 3 LLMs created")


# ============================================================
# 5. STATE
# ============================================================

class State(TypedDict):

    question: str

    # Answer generated by LLM 1
    llm1_answer: str

    # Result of LLM 2 checking
    check_result: str

    # Corrected answer from LLM 2
    llm2_answer: str

    # Final answer
    final_answer: str

    # Which LLM produced final answer
    selected_llm: str

    # Error information
    error_llm1: str
    error_llm2: str
    error_llm3: str


# ============================================================
# 6. LLM 1 NODE
# ============================================================
# LLM 1 ONLY GENERATES THE INITIAL ANSWER
# ============================================================

def llm1_generate(state: State):

    question = state["question"]

    print()
    print("🔵 LLM 1 - GENERATING ANSWER...")
    print("-" * 60)

    prompt = (
        "You are the primary AI answer generator.\n\n"
        "Answer the user's question accurately, clearly and completely.\n"
        "Do not discuss this instruction.\n\n"
        "USER QUESTION:\n"
        + question
    )

    try:

        response = llm1.invoke(prompt)

        answer = str(response.content).strip()

        if not answer:

            raise Exception(
                "LLM 1 returned an empty answer."
            )

        print("✅ LLM 1 - ANSWER GENERATED")

        return {
            "llm1_answer": answer,
            "selected_llm": "LLM 1 generated the initial answer"
        }

    except Exception as e:

        print("❌ LLM 1 FAILED")

        return {
            "error_llm1": str(e)
        }


# ============================================================
# 7. ROUTE AFTER LLM 1
# ============================================================
#
# IMPORTANT:
#
# If LLM 1 succeeds:
#       LLM 1 → LLM 2
#
# If LLM 1 fails:
#       LLM 1 → LLM 3
#
# ============================================================

def route_after_llm1(state: State):

    if state.get("llm1_answer"):

        print()
        print("➡️ LLM 1 succeeded.")
        print("➡️ Sending LLM 1 answer to LLM 2 for checking.")

        return "llm2"

    print()
    print("⚠️ LLM 1 failed.")
    print("➡️ Sending question directly to LLM 3.")

    return "llm3"


# ============================================================
# 8. LLM 2 NODE
# ============================================================
#
# LLM 2 receives:
#
# 1. Original question
# 2. LLM 1 answer
#
# It checks whether LLM 1 is correct.
#
# ============================================================

def llm2_check(state: State):

    question = state["question"]

    llm1_answer = state["llm1_answer"]

    print()
    print("🟠 LLM 2 - CHECKING LLM 1 ANSWER...")
    print("-" * 60)

    prompt = f"""
You are the verification and correction AI.

Your job is to check the answer generated by LLM 1.

USER QUESTION:
{question}

LLM 1 ANSWER:
{llm1_answer}

Now carefully verify the LLM 1 answer.

Check:
1. Is it factually correct?
2. Does it answer the question?
3. Is there any important mistake?
4. Is anything misleading?
5. Is the explanation sufficiently clear?

You MUST begin your response with exactly one of:

CORRECT

or

INCORRECT

If the answer is CORRECT:
Write:

CORRECT
Then explain briefly why it is correct.

If the answer is INCORRECT:
Write:

INCORRECT
Then provide the complete corrected answer.

Do not use both words.
"""

    try:

        response = llm2.invoke(prompt)

        result = str(response.content).strip()

        if not result:

            raise Exception(
                "LLM 2 returned an empty response."
            )

        # ----------------------------------------------------
        # Determine whether LLM 2 says correct or incorrect
        # ----------------------------------------------------

        first_line = result.splitlines()[0].strip().upper()

        if first_line.startswith("CORRECT"):

            print("✅ LLM 2 - LLM 1 ANSWER IS CORRECT")

            return {
                "check_result": "CORRECT",
                "llm2_answer": "",
                "final_answer": llm1_answer,
                "selected_llm": "LLM 1 answer verified by LLM 2"
            }

        elif first_line.startswith("INCORRECT"):

            print("❌ LLM 2 - LLM 1 ANSWER IS INCORRECT")
            print("🔧 LLM 2 - CORRECTING THE ANSWER...")

            # Remove the word INCORRECT from the beginning
            corrected = result

            lines = corrected.splitlines()

            if len(lines) > 1:

                corrected = "\n".join(
                    lines[1:]
                ).strip()

            if not corrected:

                raise Exception(
                    "LLM 2 marked the answer incorrect "
                    "but did not provide a correction."
                )

            print("✅ LLM 2 - CORRECTED ANSWER GENERATED")

            return {
                "check_result": "INCORRECT",
                "llm2_answer": corrected,
                "final_answer": corrected,
                "selected_llm": "LLM 2 corrected the answer"
            }

        else:

            raise Exception(
                "LLM 2 did not return CORRECT or INCORRECT."
            )

    except Exception as e:

        print("❌ LLM 2 FAILED")

        return {
            "error_llm2": str(e)
        }


# ============================================================
# 9. ROUTE AFTER LLM 2
# ============================================================
#
# If LLM 2 successfully checks:
#
#     CORRECT   → END
#     INCORRECT → END with corrected answer
#
# If LLM 2 fails:
#
#     → LLM 3
#
# ============================================================

def route_after_llm2(state: State):

    if state.get("final_answer"):

        print()
        print("➡️ LLM 2 completed verification.")
        print("➡️ Final answer is ready.")

        return "end"

    print()
    print("⚠️ LLM 2 failed.")
    print("➡️ Sending question to LLM 3.")

    return "llm3"


# ============================================================
# 10. LLM 3 NODE
# ============================================================
#
# LLM 3 is the emergency backup.
#
# It is used when:
#
# LLM 1 fails
# OR
# LLM 2 fails
#
# ============================================================

def llm3_backup(state: State):

    question = state["question"]

    print()
    print("🟢 LLM 3 - EMERGENCY BACKUP...")
    print("-" * 60)

    prompt = (
        "You are the emergency backup AI.\n\n"
        "The previous AI workflow was unavailable.\n"
        "Answer the user's question directly.\n"
        "Provide an accurate, complete and clear answer.\n\n"
        "USER QUESTION:\n"
        + question
    )

    try:

        response = llm3.invoke(prompt)

        answer = str(response.content).strip()

        if not answer:

            raise Exception(
                "LLM 3 returned an empty answer."
            )

        print("✅ LLM 3 - BACKUP ANSWER GENERATED")

        return {
            "final_answer": answer,
            "selected_llm": "LLM 3 - Emergency Backup"
        }

    except Exception as e:

        print("❌ LLM 3 FAILED")

        return {
            "error_llm3": str(e)
        }


# ============================================================
# 11. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(State)


# ------------------------------------------------------------
# Nodes
# ------------------------------------------------------------

builder.add_node(
    "llm1",
    llm1_generate
)

builder.add_node(
    "llm2",
    llm2_check
)

builder.add_node(
    "llm3",
    llm3_backup
)


# ------------------------------------------------------------
# START → LLM 1
# ------------------------------------------------------------

builder.add_edge(
    START,
    "llm1"
)


# ------------------------------------------------------------
# LLM 1 → LLM 2 OR LLM 3
# ------------------------------------------------------------

builder.add_conditional_edges(
    "llm1",
    route_after_llm1,
    {
        "llm2": "llm2",
        "llm3": "llm3"
    }
)


# ------------------------------------------------------------
# LLM 2 → END OR LLM 3
# ------------------------------------------------------------

builder.add_conditional_edges(
    "llm2",
    route_after_llm2,
    {
        "end": END,
        "llm3": "llm3"
    }
)


# ------------------------------------------------------------
# LLM 3 → END
# ------------------------------------------------------------

builder.add_edge(
    "llm3",
    END
)


# ------------------------------------------------------------
# Compile
# ------------------------------------------------------------

graph = builder.compile()


print()
print("=" * 70)
print("✅ LANGGRAPH COMPILED SUCCESSFULLY")
print("=" * 70)


# ============================================================
# 12. MAIN FUNCTION
# ============================================================

def ask_ai(question):

    if not question or not question.strip():

        return (
            "❌ Please enter a question.",
            "Please enter a question."
        )

    question = question.strip()

    print()
    print("=" * 70)
    print("QUESTION")
    print("=" * 70)
    print(question)
    print("=" * 70)


    # --------------------------------------------------------
    # Initial State
    # --------------------------------------------------------

    initial_state = {

        "question": question,

        "llm1_answer": "",

        "check_result": "",

        "llm2_answer": "",

        "final_answer": "",

        "selected_llm": "",

        "error_llm1": "",

        "error_llm2": "",

        "error_llm3": ""
    }


    try:

        result = graph.invoke(
            initial_state
        )


        final_answer = result.get(
            "final_answer",
            ""
        )

        selected_llm = result.get(
            "selected_llm",
            ""
        )


        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if final_answer:

            print()
            print("=" * 70)
            print("FINAL RESULT")
            print("=" * 70)

            print(
                "Selected:",
                selected_llm
            )

            print("=" * 70)


            return (
                "✅ " + selected_llm,
                final_answer
            )


        # ----------------------------------------------------
        # ALL FAILED
        # ----------------------------------------------------

        errors = []


        if result.get("error_llm1"):

            errors.append(
                "LLM 1 Error:\n"
                + result["error_llm1"]
            )


        if result.get("error_llm2"):

            errors.append(
                "LLM 2 Error:\n"
                + result["error_llm2"]
            )


        if result.get("error_llm3"):

            errors.append(
                "LLM 3 Error:\n"
                + result["error_llm3"]
            )


        error_text = (
            "❌ ALL LLMs FAILED\n\n"
            + "\n\n".join(errors)
        )


        return (
            error_text,
            "No LLM was able to provide an answer."
        )


    except Exception as e:

        return (
            "❌ Graph Error",
            str(e)
        )


# ============================================================
# 13. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Generate → Check → Correct → Backup"
) as demo:

    gr.Markdown(
        """
# 🤖 Multi-LLM Generate → Check → Correct → Backup

### Workflow

**LLM 1 — Generate**
⬇️

**LLM 2 — Check**
⬇️

**Correct?**
- ✅ YES → Final Answer
- ❌ NO → LLM 2 Corrects → Final Answer

### Emergency Backup

If **LLM 1 fails** or **LLM 2 fails**:

**LLM 3 → Final Answer**
"""
    )


    question_box = gr.Textbox(
        label="📝 Enter Your Question",
        placeholder="Example: What is artificial intelligence?",
        lines=5
    )


    ask_button = gr.Button(
        "🚀 Ask AI",
        variant="primary"
    )


    selected_output = gr.Textbox(
        label="🔀 Workflow Result",
        lines=2
    )


    answer_output = gr.Markdown(
        label="💬 Final Answer"
    )


    ask_button.click(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            selected_output,
            answer_output
        ]
    )


    question_box.submit(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            selected_output,
            answer_output
        ]
    )


# ============================================================
# 14. LAUNCH
# ============================================================

print()
print("=" * 70)
print("🚀 STARTING GRADIO")
print("=" * 70)

demo.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 81.7 MB/s eta 0:00:00
API KEY STATUS
openrouter   : ✅ Found
GROQ_API_KEY : ✅ Found
akshara      : ✅ Found
✅ 3 LLMs created

✅ LANGGRAPH COMPILED SUCCESSFULLY

🚀 STARTING GRADIO
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in l


QUESTION
what is AI?

🔵 LLM 1 - GENERATING ANSWER...
------------------------------------------------------------
✅ LLM 1 - ANSWER GENERATED

➡️ LLM 1 succeeded.
➡️ Sending LLM 1 answer to LLM 2 for checking.

🟠 LLM 2 - CHECKING LLM 1 ANSWER...
------------------------------------------------------------
✅ LLM 2 - LLM 1 ANSWER IS CORRECT

➡️ LLM 2 completed verification.
➡️ Final answer is ready.

FINAL RESULT
Selected: LLM 1 answer verified by LLM 2

QUESTION
TEST_LLM3_BACKUP

🔵 LLM 1 - GENERATING ANSWER...
------------------------------------------------------------
✅ LLM 1 - ANSWER GENERATED

➡️ LLM 1 succeeded.
➡️ Sending LLM 1 answer to LLM 2 for checking.

🟠 LLM 2 - CHECKING LLM 1 ANSWER...
------------------------------------------------------------
❌ LLM 2 - LLM 1 ANSWER IS INCORRECT
🔧 LLM 2 - CORRECTING THE ANSWER...
✅ LLM 2 - CORRECTED ANSWER GENERATED

➡️ LLM 2 completed verification.
➡️ Final answer is ready.

FINAL RESULT
Selected: LLM 2 corrected the answer

QUESTION
FOR